<a href="https://colab.research.google.com/github/islam37/S-SLTE-Thesis/blob/main/notebooks/10_s_slte_foundation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

from pathlib import Path
import json
import hashlib
import pandas as pd
import torch
from google.colab import drive

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")

# Load frozen research protocols.
with (BASE / "configs/holdout_partition_manifest_v1.json").open() as f:
    partition_manifest = json.load(f)

with (BASE / "outputs/llm_main_v1/full_annotation_audit_v1.json").open() as f:
    annotation_audit = json.load(f)

assert annotation_audit["status"] == "FULL_CHECKPOINT_AUDIT_PASSED"
assert annotation_audit["total_records"] == 15621
assert annotation_audit["invalid_predictions"] == 0

# Load clean reference partitions and text-only training data.
anchor_df = pd.read_csv(BASE / "data/splits/anchor.csv")
dev_df = pd.read_csv(BASE / "data/splits/development.csv")

source_training = pd.read_csv(
    BASE / "data/splits/noisy_training.csv",
    usecols=["record_id", "text"]
)

llm_annotations = pd.read_csv(
    BASE / "outputs/llm_main_v1/training_annotations_v1.csv"
)

def id_fingerprint(ids):
    payload = "\n".join(
        str(int(x)) for x in sorted(ids)
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def text_fingerprint(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values("record_id").itertuples(index=False):
        digest.update(str(int(row.record_id)).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(row.text).encode("utf-8"))
        digest.update(b"\0")

    return digest.hexdigest()

def verify_partition(frame, partition):
    spec = partition_manifest["partitions"][partition]
    ids = set(map(int, spec["record_ids"]))

    assert len(frame) == spec["count"]
    assert frame["record_id"].is_unique
    assert set(frame["record_id"].astype(int)) == ids
    assert id_fingerprint(ids) == spec["id_sha256"]
    assert text_fingerprint(frame[["record_id", "text"]]) == (
        spec["raw_text_sha256"]
    )

verify_partition(anchor_df, "anchor")
verify_partition(dev_df, "development")

training_ids = set(map(
    int,
    partition_manifest["partitions"]["training"]["record_ids"]
))

training_text = source_training.loc[
    source_training["record_id"].isin(training_ids)
].copy()

verify_partition(training_text, "training")

assert llm_annotations["record_id"].is_unique
assert set(llm_annotations["record_id"].astype(int)) == training_ids

training_df = training_text.merge(
    llm_annotations,
    on="record_id",
    validate="one_to_one"
)

assert len(training_df) == 13621
assert training_df["llm_label"].isin([0, 1, 2, 3]).all()
assert "label" not in training_df.columns

for frame in [anchor_df, dev_df]:
    assert {"record_id", "text", "label"}.issubset(frame.columns)
    assert frame["label"].isin([0, 1, 2, 3]).all()

assert set(anchor_df["record_id"]).isdisjoint(training_ids)
assert set(dev_df["record_id"]).isdisjoint(training_ids)
assert set(anchor_df["record_id"]).isdisjoint(
    set(dev_df["record_id"])
)

assert torch.cuda.is_available(), "Enable T4 GPU."

print("STAGE 11 — FOUNDATION DATA VALIDATION PASSED")
print("Clean anchor:", len(anchor_df))
print("Development:", len(dev_df))
print("Noisy training:", len(training_df))
print("Frozen fingerprints: VERIFIED")
print("Training reference labels accessed: NO")
print("Holdout reference labels accessed: NO")
print("GPU:", torch.cuda.get_device_name(0))
print("Embedding generation: NOT STARTED")


Mounted at /content/drive
STAGE 11 — FOUNDATION DATA VALIDATION PASSED
Clean anchor: 2000
Development: 1800
Noisy training: 13621
Frozen fingerprints: VERIFIED
Training reference labels accessed: NO
Holdout reference labels accessed: NO
GPU: Tesla T4
Embedding generation: NOT STARTED


In [2]:

%pip install -q "sentence-transformers==5.7.0" "transformers==5.20.1"

import json
import importlib.metadata as metadata

import numpy as np
import torch
from huggingface_hub import HfApi
from sentence_transformers import SentenceTransformer

MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
PROTOCOL_PATH = BASE / "configs/s_slte_embedding_protocol_v1.json"

assert torch.cuda.is_available()
assert metadata.version("sentence-transformers") == "5.7.0"
assert metadata.version("transformers") == "5.20.1"

# Reuse the frozen revision if the protocol already exists.
if PROTOCOL_PATH.exists():
    with PROTOCOL_PATH.open(encoding="utf-8") as f:
        protocol = json.load(f)

    assert protocol["model_id"] == MODEL_ID
    MODEL_REVISION = protocol["model_revision_sha"]
else:
    MODEL_REVISION = HfApi().model_info(MODEL_ID).sha

assert len(MODEL_REVISION) == 40

embedding_model = SentenceTransformer(
    MODEL_ID,
    revision=MODEL_REVISION,
    device="cuda",
    trust_remote_code=False
)

embedding_model.eval()

EMBEDDING_DIM = embedding_model.get_sentence_embedding_dimension()
MAX_SEQUENCE_LENGTH = embedding_model.max_seq_length

assert EMBEDDING_DIM == 384
assert MAX_SEQUENCE_LENGTH == 256

# Small inference test before processing the complete dataset.
sample_vectors = embedding_model.encode(
    training_df["text"].head(4).astype(str).tolist(),
    batch_size=4,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
)

assert sample_vectors.shape == (4, 384)
assert np.isfinite(sample_vectors).all()
assert np.allclose(
    np.linalg.norm(sample_vectors, axis=1),
    1.0,
    atol=1e-5
)

expected_protocol = {
    "protocol_version": "v1",
    "model_id": MODEL_ID,
    "model_revision_sha": MODEL_REVISION,
    "embedding_dimension": 384,
    "max_sequence_length": 256,
    "normalize_embeddings": True,
    "input_text_policy": "ORIGINAL_TEXT_UNMODIFIED",
    "sentence_transformers_version": "5.7.0",
    "transformers_version": "5.20.1"
}

# Create the experiment protocol without overwriting existing data.
if PROTOCOL_PATH.exists():
    assert protocol == expected_protocol
    protocol_status = "EXISTING_VERIFIED"
else:
    with PROTOCOL_PATH.open("x", encoding="utf-8") as f:
        json.dump(expected_protocol, f, indent=2)
    protocol_status = "NEW_SAVED"

print("STAGE 11 — SENTENCE-BERT VALIDATION PASSED")
print("Model:", MODEL_ID)
print("Revision:", MODEL_REVISION)
print("GPU:", torch.cuda.get_device_name(0))
print("Embedding dimension:", EMBEDDING_DIM)
print("Maximum sequence length:", MAX_SEQUENCE_LENGTH)
print("Sample embedding shape:", sample_vectors.shape)
print("Normalized embeddings: VERIFIED")
print("Protocol:", protocol_status)
print("Reference labels used: NO")
print("Full embedding generation: NOT STARTED")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_1277/1243744824.py:39: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMBEDDING_DIM = embedding_model.get_sentence_embedding_dimension()


STAGE 11 — SENTENCE-BERT VALIDATION PASSED
Model: sentence-transformers/all-MiniLM-L6-v2
Revision: 1110a243fdf4706b3f48f1d95db1a4f5529b4d41
GPU: Tesla T4
Embedding dimension: 384
Maximum sequence length: 256
Sample embedding shape: (4, 384)
Normalized embeddings: VERIFIED
Protocol: NEW_SAVED
Reference labels used: NO
Full embedding generation: NOT STARTED


In [3]:

import os
import json
import hashlib
from pathlib import Path

import numpy as np
import torch

# Save embeddings under a versioned, private Drive directory.
EMBEDDING_DIR = BASE / "outputs" / "s_slte_embeddings_v1"
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 64

assert embedding_model.device.type == "cuda"
assert EMBEDDING_DIM == 384
assert MAX_SEQUENCE_LENGTH == 256
assert PROTOCOL_PATH.exists()

with PROTOCOL_PATH.open(encoding="utf-8") as f:
    saved_protocol = json.load(f)

assert saved_protocol["model_id"] == MODEL_ID
assert saved_protocol["model_revision_sha"] == MODEL_REVISION
assert saved_protocol["normalize_embeddings"] is True

partitions = {
    "anchor": anchor_df,
    "development": dev_df,
    "training": training_df
}

expected_counts = {
    "anchor": 2000,
    "development": 1800,
    "training": 13621
}

def verify_embedding_file(path, expected_ids):
    with np.load(path, allow_pickle=False) as saved:
        ids = saved["record_ids"]
        vectors = saved["embeddings"]

    assert np.array_equal(ids, expected_ids)
    assert vectors.shape == (len(expected_ids), 384)
    assert vectors.dtype == np.float32
    assert np.isfinite(vectors).all()

    norms = np.linalg.norm(vectors, axis=1)
    assert np.allclose(norms, 1.0, atol=1e-4)

    return vectors.shape

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

artifact_manifest = {
    "protocol_version": "v1",
    "model_id": MODEL_ID,
    "model_revision_sha": MODEL_REVISION,
    "embedding_dimension": EMBEDDING_DIM,
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "normalized": True,
    "batch_size": BATCH_SIZE,
    "input_policy": "ORIGINAL_TEXT_UNMODIFIED",
    "partitions": {}
}

for partition, frame in partitions.items():

    data = (
        frame[["record_id", "text"]]
        .sort_values("record_id")
        .reset_index(drop=True)
    )

    assert len(data) == expected_counts[partition]
    assert data["record_id"].is_unique
    assert data["text"].notna().all()

    verify_partition(data, partition)

    ids = data["record_id"].to_numpy(dtype=np.int64)
    path = EMBEDDING_DIR / f"{partition}_embeddings_v1.npz"

    if path.exists():
        status = "EXISTING_VERIFIED"

    else:
        print(f"\nGenerating embeddings: {partition}")

        texts = data["text"].astype(str).tolist()

        vectors = embedding_model.encode(
            texts,
            batch_size=BATCH_SIZE,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True
        ).astype(np.float32)

        assert vectors.shape == (len(ids), 384)
        assert np.isfinite(vectors).all()
        assert np.allclose(
            np.linalg.norm(vectors, axis=1),
            1.0,
            atol=1e-4
        )

        temporary_path = path.with_suffix(".npz.tmp")

        with temporary_path.open("wb") as f:
            np.savez_compressed(
                f,
                record_ids=ids,
                embeddings=vectors
            )

        os.replace(temporary_path, path)
        status = "NEW_SAVED"

        del vectors

    shape = verify_embedding_file(path, ids)

    artifact_manifest["partitions"][partition] = {
        "count": len(ids),
        "shape": list(shape),
        "filename": path.name,
        "id_sha256": id_fingerprint(ids),
        "file_sha256": sha256_file(path)
    }

    print(
        f"{partition}: {status} | "
        f"Shape: {shape} | "
        f"Integrity: PASSED"
    )

manifest_path = EMBEDDING_DIR / "embedding_manifest_v1.json"

if manifest_path.exists():
    with manifest_path.open(encoding="utf-8") as f:
        existing_manifest = json.load(f)

    assert existing_manifest == artifact_manifest
    manifest_status = "EXISTING_VERIFIED"

else:
    with manifest_path.open("x", encoding="utf-8") as f:
        json.dump(artifact_manifest, f, indent=2)

    manifest_status = "NEW_SAVED"

print("\nSTAGE 11 — EMBEDDING GENERATION SUMMARY")
print("Total:", sum(expected_counts.values()))
print("Dimension:", EMBEDDING_DIM)
print("Partitions:", len(partitions))
print("Model revision: VERIFIED")
print("Normalized vectors: VERIFIED")
print("Frozen partition integrity: VERIFIED")
print("Embedding manifest:", manifest_status)
print("Training reference labels accessed: NO")
print("Holdout reference labels accessed: NO")
print("Files:", EMBEDDING_DIR)
print("\nSTATUS: EMBEDDINGS GENERATED AND VERIFIED")



Generating embeddings: anchor


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

anchor: NEW_SAVED | Shape: (2000, 384) | Integrity: PASSED

Generating embeddings: development


Batches:   0%|          | 0/29 [00:00<?, ?it/s]

development: NEW_SAVED | Shape: (1800, 384) | Integrity: PASSED

Generating embeddings: training


Batches:   0%|          | 0/213 [00:00<?, ?it/s]

training: NEW_SAVED | Shape: (13621, 384) | Integrity: PASSED

STAGE 11 — EMBEDDING GENERATION SUMMARY
Total: 17421
Dimension: 384
Partitions: 3
Model revision: VERIFIED
Normalized vectors: VERIFIED
Frozen partition integrity: VERIFIED
Embedding manifest: NEW_SAVED
Training reference labels accessed: NO
Holdout reference labels accessed: NO
Files: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_embeddings_v1

STATUS: EMBEDDINGS GENERATED AND VERIFIED


In [4]:

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

assert MAX_SEQUENCE_LENGTH == 256
assert EMBEDDING_DIM == 384

TOKEN_BATCH_SIZE = 256
tokenizer = embedding_model.tokenizer

partitions = {
    "anchor": anchor_df,
    "development": dev_df,
    "training": training_df
}

audit_rows = []
partition_results = {}

for partition, frame in partitions.items():
    data = (
        frame[["record_id", "text"]]
        .sort_values("record_id")
        .reset_index(drop=True)
    )

    verify_partition(data, partition)

    texts = data["text"].astype(str).tolist()
    lengths = []

    for start in range(0, len(texts), TOKEN_BATCH_SIZE):
        batch = texts[start:start + TOKEN_BATCH_SIZE]

        encoded = tokenizer(
            batch,
            add_special_tokens=True,
            truncation=False,
            padding=False,
            return_length=True,
            verbose=False
        )

        lengths.extend(encoded["length"])

    lengths = np.asarray(lengths, dtype=np.int32)

    assert len(lengths) == len(data)
    assert np.all(lengths > 0)

    truncated = lengths > MAX_SEQUENCE_LENGTH
    truncated_count = int(truncated.sum())

    result = {
        "partition": partition,
        "total_records": len(data),
        "truncated_records": truncated_count,
        "truncation_rate_percent": round(
            100 * truncated_count / len(data), 4
        ),
        "mean_tokens": round(float(lengths.mean()), 2),
        "median_tokens": float(np.median(lengths)),
        "p95_tokens": float(np.percentile(lengths, 95)),
        "max_tokens": int(lengths.max()),
        "max_sequence_length": MAX_SEQUENCE_LENGTH,
        "record_id_sha256": id_fingerprint(
            data["record_id"].to_numpy()
        )
    }

    partition_results[partition] = result
    audit_rows.append(result)

audit_df = pd.DataFrame(audit_rows)

total_records = sum(
    item["total_records"]
    for item in partition_results.values()
)

total_truncated = sum(
    item["truncated_records"]
    for item in partition_results.values()
)

assert total_records == 17421

audit_report = {
    "protocol_version": "v1",
    "model_id": MODEL_ID,
    "model_revision_sha": MODEL_REVISION,
    "tokenizer_class": type(tokenizer).__name__,
    "token_count_includes_special_tokens": True,
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "input_policy": "ORIGINAL_TEXT_UNMODIFIED",
    "total_records": total_records,
    "total_truncated_records": total_truncated,
    "overall_truncation_rate_percent": round(
        100 * total_truncated / total_records, 4
    ),
    "partitions": partition_results,
    "reference_labels_accessed": False
}

audit_path = EMBEDDING_DIR / "truncation_audit_v1.json"
payload = json.dumps(
    audit_report,
    indent=2,
    ensure_ascii=False
) + "\n"

if audit_path.exists():
    assert audit_path.read_text(encoding="utf-8") == payload
    status = "EXISTING_VERIFIED"
else:
    temporary_path = audit_path.with_suffix(".json.tmp")

    with temporary_path.open("w", encoding="utf-8") as f:
        f.write(payload)

    os.replace(temporary_path, audit_path)
    status = "NEW_SAVED"

print("STAGE 11 — TOKEN TRUNCATION AUDIT")
display(audit_df)

print("Total records:", total_records)
print("Records exceeding 256 tokens:", total_truncated)
print(
    "Overall truncation rate:",
    f"{audit_report['overall_truncation_rate_percent']:.2f}%"
)
print("Audit artifact:", status)
print("Saved path:", audit_path)
print("Reference labels accessed: NO")
print("\nSTATUS: TOKEN TRUNCATION AUDIT PASSED")


STAGE 11 — TOKEN TRUNCATION AUDIT


,partition,total_records,truncated_records,truncation_rate_percent,mean_tokens,median_tokens,p95_tokens,max_tokens,max_sequence_length,record_id_sha256
0,anchor,2000,1,0.0500,52.87,50.0,79.00,379,256,ac474666d787836fec02b33510d4ce0b04074297fb1067...
1,development,1800,2,0.1111,53.25,51.0,81.05,348,256,8d847b8c95dbc031d7826deeb0ac7efa77dbb08a5b598d...
2,training,13621,6,0.0440,53.34,51.0,83.00,351,256,ce13626940296ed9158f4dc116580987f404114f66d42f...


Total records: 17421
Records exceeding 256 tokens: 9
Overall truncation rate: 0.05%
Audit artifact: NEW_SAVED
Saved path: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_embeddings_v1/truncation_audit_v1.json
Reference labels accessed: NO

STATUS: TOKEN TRUNCATION AUDIT PASSED


In [5]:

import os
import json
import hashlib
from pathlib import Path

import numpy as np

K = 20
QUERY_BATCH_SIZE = 256
KNN_DIR = BASE / "outputs" / "s_slte_knn_v1"
KNN_DIR.mkdir(parents=True, exist_ok=True)

# Verify the frozen embedding artifacts before using them.
with (
    EMBEDDING_DIR / "embedding_manifest_v1.json"
).open(encoding="utf-8") as f:
    embedding_manifest = json.load(f)

assert embedding_manifest["model_id"] == MODEL_ID
assert embedding_manifest["model_revision_sha"] == MODEL_REVISION
assert embedding_manifest["embedding_dimension"] == 384
assert embedding_manifest["normalized"] is True

def file_sha256(path):
    digest = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

def load_verified_embeddings(partition, expected_frame):
    path = EMBEDDING_DIR / f"{partition}_embeddings_v1.npz"
    spec = embedding_manifest["partitions"][partition]

    assert file_sha256(path) == spec["file_sha256"]

    with np.load(path, allow_pickle=False) as data:
        ids = data["record_ids"].copy()
        vectors = data["embeddings"].copy()

    expected_ids = (
        expected_frame["record_id"]
        .sort_values()
        .to_numpy(dtype=np.int64)
    )

    assert np.array_equal(ids, expected_ids)
    assert vectors.shape == (len(ids), 384)
    assert vectors.dtype == np.float32
    assert np.isfinite(vectors).all()
    assert np.allclose(
        np.linalg.norm(vectors, axis=1), 1, atol=1e-4
    )

    return ids, vectors

anchor_ids, anchor_vectors = load_verified_embeddings(
    "anchor", anchor_df
)

# Trusted anchor labels are allowed as neighborhood evidence.
anchor_sorted = (
    anchor_df.sort_values("record_id").reset_index(drop=True)
)

anchor_labels = anchor_sorted["label"].to_numpy(dtype=np.int64)

assert np.array_equal(
    anchor_sorted["record_id"].to_numpy(dtype=np.int64),
    anchor_ids
)
assert np.isin(anchor_labels, [0, 1, 2, 3]).all()
assert K <= len(anchor_ids)

def validate_knn(path, query_ids):
    with np.load(path, allow_pickle=False) as saved:
        stored_queries = saved["query_record_ids"].copy()
        neighbor_ids = saved["anchor_record_ids"].copy()
        neighbor_labels = saved["anchor_labels"].copy()
        scores = saved["cosine_similarity"].copy()

    expected_shape = (len(query_ids), K)

    assert np.array_equal(stored_queries, query_ids)
    assert neighbor_ids.shape == expected_shape
    assert neighbor_labels.shape == expected_shape
    assert scores.shape == expected_shape

    assert np.isfinite(scores).all()
    assert np.all(scores >= -1.0001)
    assert np.all(scores <= 1.0001)
    assert np.all(np.diff(scores, axis=1) <= 1e-5)

    anchor_position = {
        int(record_id): index
        for index, record_id in enumerate(anchor_ids)
    }

    indices = np.array([
        [anchor_position[int(rid)] for rid in row]
        for row in neighbor_ids
    ], dtype=np.int64)

    assert np.array_equal(
        neighbor_labels, anchor_labels[indices]
    )

    assert all(
        len(set(row.tolist())) == K
        for row in neighbor_ids
    )

    # Independently verify cosine scores for sampled queries.
    sample_indices = np.unique(
        np.linspace(0, len(query_ids) - 1, 10, dtype=int)
    )

    return indices, scores, sample_indices

knn_manifest = {
    "protocol_version": "v1",
    "model_id": MODEL_ID,
    "model_revision_sha": MODEL_REVISION,
    "similarity": "COSINE_NORMALIZED_DOT_PRODUCT",
    "k": K,
    "neighbor_pool": "CLEAN_ANCHOR_ONLY",
    "tie_break": "ASCENDING_ANCHOR_RECORD_ID",
    "selection_policy": "TOP_K_EXACT",
    "reference_labels_policy": (
        "TRUSTED_ANCHOR_LABELS_ONLY"
    ),
    "partitions": {}
}

for partition, frame in [
    ("development", dev_df),
    ("training", training_df)
]:
    query_ids, query_vectors = load_verified_embeddings(
        partition, frame
    )

    assert set(query_ids).isdisjoint(set(anchor_ids))

    output_path = (
        KNN_DIR / f"{partition}_anchor_knn_k{K}_v1.npz"
    )

    if output_path.exists():
        status = "EXISTING_VERIFIED"

    else:
        all_neighbor_ids = []
        all_neighbor_labels = []
        all_scores = []

        for start in range(0, len(query_ids), QUERY_BATCH_SIZE):
            query_batch = query_vectors[
                start:start + QUERY_BATCH_SIZE
            ]

            # All embeddings are L2-normalized.
            similarities = query_batch @ anchor_vectors.T

            # Anchor IDs are sorted: stable sort resolves score ties.
            order = np.argsort(
                -similarities,
                axis=1,
                kind="stable"
            )[:, :K]

            selected_scores = np.take_along_axis(
                similarities, order, axis=1
            )

            all_neighbor_ids.append(anchor_ids[order])
            all_neighbor_labels.append(anchor_labels[order])
            all_scores.append(selected_scores)

        neighbor_ids = np.concatenate(all_neighbor_ids)
        neighbor_labels = np.concatenate(all_neighbor_labels)
        scores = np.concatenate(all_scores).astype(np.float32)

        assert neighbor_ids.shape == (len(query_ids), K)
        assert scores.shape == (len(query_ids), K)

        temporary_path = output_path.with_suffix(".npz.tmp")

        with temporary_path.open("wb") as f:
            np.savez_compressed(
                f,
                query_record_ids=query_ids,
                anchor_record_ids=neighbor_ids,
                anchor_labels=neighbor_labels,
                cosine_similarity=scores
            )

        os.replace(temporary_path, output_path)
        status = "NEW_SAVED"

    indices, scores, sample_indices = validate_knn(
        output_path, query_ids
    )

    computed_scores = np.sum(
        query_vectors[sample_indices, None, :]
        * anchor_vectors[indices[sample_indices]],
        axis=2
    )

    assert np.allclose(
        scores[sample_indices],
        computed_scores,
        atol=1e-5
    )

    knn_manifest["partitions"][partition] = {
        "count": len(query_ids),
        "filename": output_path.name,
        "shape": [len(query_ids), K],
        "query_ids_sha256": id_fingerprint(query_ids),
        "file_sha256": file_sha256(output_path)
    }

    print(
        f"{partition}: {status} | "
        f"Queries: {len(query_ids)} | "
        f"Neighbors per query: {K} | "
        "Integrity: PASSED"
    )

manifest_path = KNN_DIR / "anchor_knn_manifest_v1.json"
manifest_content = json.dumps(knn_manifest, indent=2) + "\n"

if manifest_path.exists():
    assert manifest_path.read_text(encoding="utf-8") == (
        manifest_content
    )
    manifest_status = "EXISTING_VERIFIED"

else:
    temporary = manifest_path.with_suffix(".json.tmp")

    with temporary.open("w", encoding="utf-8") as f:
        f.write(manifest_content)

    os.replace(temporary, manifest_path)
    manifest_status = "NEW_SAVED"

print("\nSTAGE 11 — SEMANTIC KNN SUMMARY")
print("Clean anchor pool:", len(anchor_ids))
print("Development queries:", len(dev_df))
print("Training queries:", len(training_df))
print("Neighbors per query:", K)
print("Similarity: COSINE")
print("Neighbor ranking: EXACT")
print("Anchor labels used: YES — TRUSTED")
print("Development labels used for ranking: NO")
print("Training reference labels accessed: NO")
print("Holdout reference labels accessed: NO")
print("Manifest:", manifest_status)
print("Output directory:", KNN_DIR)
print("\nSTATUS: SEMANTIC KNN GENERATED AND VERIFIED")


development: NEW_SAVED | Queries: 1800 | Neighbors per query: 20 | Integrity: PASSED
training: NEW_SAVED | Queries: 13621 | Neighbors per query: 20 | Integrity: PASSED

STAGE 11 — SEMANTIC KNN SUMMARY
Clean anchor pool: 2000
Development queries: 1800
Training queries: 13621
Neighbors per query: 20
Similarity: COSINE
Neighbor ranking: EXACT
Anchor labels used: YES — TRUSTED
Development labels used for ranking: NO
Training reference labels accessed: NO
Holdout reference labels accessed: NO
Manifest: NEW_SAVED
Output directory: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_knn_v1

STATUS: SEMANTIC KNN GENERATED AND VERIFIED


In [6]:

import os
import json
import hashlib

import numpy as np
import pandas as pd

NUM_CLASSES = 4
K = 20

EVIDENCE_DIR = BASE / "outputs" / "s_slte_evidence_v1"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

with (
    KNN_DIR / "anchor_knn_manifest_v1.json"
).open(encoding="utf-8") as f:
    knn_manifest = json.load(f)

assert knn_manifest["model_id"] == MODEL_ID
assert knn_manifest["model_revision_sha"] == MODEL_REVISION
assert knn_manifest["k"] == K
assert knn_manifest["neighbor_pool"] == "CLEAN_ANCHOR_ONLY"

def compute_neighborhood_evidence(partition, frame):
    knn_path = (
        KNN_DIR / f"{partition}_anchor_knn_k{K}_v1.npz"
    )

    assert file_sha256(knn_path) == (
        knn_manifest["partitions"][partition]["file_sha256"]
    )

    with np.load(knn_path, allow_pickle=False) as data:
        query_ids = data["query_record_ids"].copy()
        neighbor_labels = data["anchor_labels"].copy()
        similarities = data["cosine_similarity"].copy()

    expected_ids = (
        frame["record_id"]
        .sort_values()
        .to_numpy(dtype=np.int64)
    )

    assert np.array_equal(query_ids, expected_ids)
    assert neighbor_labels.shape == (len(query_ids), K)
    assert similarities.shape == (len(query_ids), K)
    assert np.isin(neighbor_labels, range(NUM_CLASSES)).all()
    assert np.isfinite(similarities).all()

    weights = np.maximum(
        similarities.astype(np.float64), 0.0
    )
    total_weights = weights.sum(axis=1)
    has_evidence = total_weights > 0

    weighted_class_totals = np.zeros(
        (len(query_ids), NUM_CLASSES), dtype=np.float64
    )

    vote_counts = np.zeros(
        (len(query_ids), NUM_CLASSES), dtype=np.int32
    )

    for class_id in range(NUM_CLASSES):
        matches = neighbor_labels == class_id

        weighted_class_totals[:, class_id] = (
            weights * matches
        ).sum(axis=1)

        vote_counts[:, class_id] = matches.sum(axis=1)

    probabilities = np.divide(
        weighted_class_totals,
        total_weights[:, None],
        out=np.zeros_like(weighted_class_totals),
        where=has_evidence[:, None]
    )

    assert np.allclose(
        probabilities[has_evidence].sum(axis=1),
        1.0, atol=1e-7
    )
    assert np.all(probabilities >= 0)
    assert np.all(probabilities <= 1.000001)
    assert np.all(vote_counts.sum(axis=1) == K)

    top_label = probabilities.argmax(axis=1)
    top_label = np.where(has_evidence, top_label, -1)

    sorted_probabilities = np.sort(
        probabilities, axis=1
    )

    margin = (
        sorted_probabilities[:, -1]
        - sorted_probabilities[:, -2]
    )

    result = pd.DataFrame({
        "record_id": query_ids,
        "has_positive_similarity": has_evidence,
        "neighbor_top_label": top_label,
        "neighbor_top_support": probabilities.max(axis=1),
        "neighbor_support_margin": margin,
        "mean_cosine_similarity": similarities.mean(axis=1),
        "max_cosine_similarity": similarities.max(axis=1),
        "top_class_vote_count": vote_counts.max(axis=1)
    })

    for class_id in range(NUM_CLASSES):
        result[f"support_class_{class_id}"] = (
            probabilities[:, class_id]
        )

    if partition == "training":
        assert "label" not in frame.columns

        llm_labels = (
            frame.sort_values("record_id")["llm_label"]
            .to_numpy(dtype=np.int64)
        )

        assert np.isin(llm_labels, range(NUM_CLASSES)).all()

        result["llm_label"] = llm_labels
        result["llm_label_support"] = probabilities[
            np.arange(len(query_ids)), llm_labels
        ]

        result["neighbor_llm_agreement"] = (
            (top_label == llm_labels) & has_evidence
        )

    assert result["record_id"].is_unique
    assert len(result) == len(frame)

    return result

summary = {}

for partition, frame in [
    ("development", dev_df),
    ("training", training_df)
]:
    evidence = compute_neighborhood_evidence(
        partition, frame
    )

    output_path = (
        EVIDENCE_DIR / f"{partition}_neighbor_evidence_v1.csv"
    )

    csv_content = evidence.to_csv(
        index=False,
        float_format="%.8f",
        lineterminator="\n"
    )

    if output_path.exists():
        assert output_path.read_text(
            encoding="utf-8"
        ) == csv_content
        status = "EXISTING_VERIFIED"
    else:
        temporary = output_path.with_suffix(".csv.tmp")
        temporary.write_text(csv_content, encoding="utf-8")
        os.replace(temporary, output_path)
        status = "NEW_SAVED"

    summary[partition] = {
        "records": len(evidence),
        "no_positive_similarity": int(
            (~evidence["has_positive_similarity"]).sum()
        ),
        "file": output_path.name,
        "sha256": file_sha256(output_path)
    }

    if partition == "training":
        summary[partition]["neighbor_llm_agreements"] = int(
            evidence["neighbor_llm_agreement"].sum()
        )
        summary[partition]["neighbor_llm_disagreements"] = int(
            (
                evidence["has_positive_similarity"]
                & ~evidence["neighbor_llm_agreement"]
            ).sum()
        )

    print(
        f"{partition}: {status} | "
        f"Records: {len(evidence)} | "
        f"Without positive similarity: "
        f"{summary[partition]['no_positive_similarity']}"
    )

evidence_manifest = {
    "protocol_version": "v1",
    "model_id": MODEL_ID,
    "model_revision_sha": MODEL_REVISION,
    "k": K,
    "weighting": "MAX_COSINE_ZERO",
    "neighbor_pool": "CLEAN_ANCHOR_ONLY",
    "score_type": "UNCALIBRATED_NEIGHBOR_SUPPORT",
    "label_correction_performed": False,
    "training_reference_labels_accessed": False,
    "holdout_reference_labels_accessed": False,
    "partitions": summary
}

manifest_path = EVIDENCE_DIR / "evidence_manifest_v1.json"
manifest_content = json.dumps(
    evidence_manifest, indent=2
) + "\n"

if manifest_path.exists():
    assert manifest_path.read_text(
        encoding="utf-8"
    ) == manifest_content
    manifest_status = "EXISTING_VERIFIED"
else:
    temporary = manifest_path.with_suffix(".json.tmp")
    temporary.write_text(
        manifest_content, encoding="utf-8"
    )
    os.replace(temporary, manifest_path)
    manifest_status = "NEW_SAVED"

print("\nSTAGE 11 — NEIGHBORHOOD EVIDENCE SUMMARY")
print("Development records:", summary["development"]["records"])
print("Training records:", summary["training"]["records"])
print(
    "Neighbor–LLM agreements:",
    summary["training"]["neighbor_llm_agreements"]
)
print(
    "Neighbor–LLM disagreements:",
    summary["training"]["neighbor_llm_disagreements"]
)
print("Evidence manifest:", manifest_status)
print("Correction performed: NO")
print("Reference-label leakage: NONE")
print("\nSTATUS: NEIGHBORHOOD EVIDENCE GENERATED AND VERIFIED")


development: NEW_SAVED | Records: 1800 | Without positive similarity: 0
training: NEW_SAVED | Records: 13621 | Without positive similarity: 0

STAGE 11 — NEIGHBORHOOD EVIDENCE SUMMARY
Development records: 1800
Training records: 13621
Neighbor–LLM agreements: 11031
Neighbor–LLM disagreements: 2590
Evidence manifest: NEW_SAVED
Correction performed: NO
Reference-label leakage: NONE

STATUS: NEIGHBORHOOD EVIDENCE GENERATED AND VERIFIED


In [7]:

import os
import json
import hashlib

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# Load and independently verify the saved evidence.
EVIDENCE_DIR = BASE / "outputs" / "s_slte_evidence_v1"

with (EVIDENCE_DIR / "evidence_manifest_v1.json").open(
    encoding="utf-8"
) as f:
    evidence_manifest = json.load(f)

assert evidence_manifest["k"] == 20
assert evidence_manifest["neighbor_pool"] == "CLEAN_ANCHOR_ONLY"
assert evidence_manifest["label_correction_performed"] is False

evidence_path = EVIDENCE_DIR / "development_neighbor_evidence_v1.csv"

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

assert sha256_file(evidence_path) == (
    evidence_manifest["partitions"]["development"]["sha256"]
)

evidence = pd.read_csv(evidence_path)

assert len(evidence) == 1800
assert evidence["record_id"].is_unique
assert evidence["has_positive_similarity"].all()

# Clean development labels are permitted for diagnostic evaluation.
dev_labels = dev_df[["record_id", "label"]].copy()

evaluation = evidence.merge(
    dev_labels,
    on="record_id",
    how="inner",
    validate="one_to_one"
).sort_values("record_id").reset_index(drop=True)

assert len(evaluation) == 1800
assert evaluation["label"].isin(range(4)).all()
assert evaluation["neighbor_top_label"].isin(range(4)).all()

support_columns = [
    f"support_class_{class_id}"
    for class_id in range(4)
]

support_matrix = evaluation[support_columns].to_numpy()

assert np.isfinite(support_matrix).all()
assert np.allclose(
    support_matrix.sum(axis=1), 1.0, atol=1e-6
)
assert np.allclose(
    support_matrix.max(axis=1),
    evaluation["neighbor_top_support"],
    atol=1e-6
)

y_true = evaluation["label"].to_numpy(dtype=int)
y_pred = evaluation["neighbor_top_label"].to_numpy(dtype=int)

accuracy = float(accuracy_score(y_true, y_pred))
macro_f1 = float(
    f1_score(y_true, y_pred, labels=[0, 1, 2, 3],
             average="macro", zero_division=0)
)

class_names = ["World", "Sports", "Business", "Sci/Tech"]

class_results = []

for class_id, class_name in enumerate(class_names):
    mask = y_true == class_id

    class_results.append({
        "class_id": class_id,
        "class_name": class_name,
        "records": int(mask.sum()),
        "correct": int((y_pred[mask] == y_true[mask]).sum()),
        "accuracy": round(
            float((y_pred[mask] == y_true[mask]).mean()), 6
        )
    })

# Predefined thresholds for descriptive coverage/accuracy analysis.
threshold_results = []

for threshold in [0.50, 0.60, 0.70, 0.80, 0.90]:
    accepted = (
        evaluation["neighbor_top_support"].to_numpy()
        >= threshold
    )

    count = int(accepted.sum())

    threshold_results.append({
        "minimum_top_support": threshold,
        "selected_records": count,
        "coverage": round(count / len(evaluation), 6),
        "accuracy_if_selected": (
            round(
                float(accuracy_score(
                    y_true[accepted], y_pred[accepted]
                )), 6
            )
            if count else None
        )
    })

confusion = confusion_matrix(
    y_true, y_pred, labels=[0, 1, 2, 3]
)

report = {
    "protocol_version": "v1",
    "evaluation_partition": "development",
    "records": 1800,
    "neighbor_pool": "CLEAN_ANCHOR_ONLY",
    "k": 20,
    "accuracy": round(accuracy, 6),
    "macro_f1": round(macro_f1, 6),
    "class_metrics": class_results,
    "confusion_matrix": confusion.tolist(),
    "confidence_threshold_diagnostics": threshold_results,
    "threshold_selected": False,
    "label_correction_performed": False,
    "training_reference_labels_accessed": False,
    "holdout_reference_labels_accessed": False,
    "source_evidence_sha256": sha256_file(evidence_path)
}

report_path = EVIDENCE_DIR / "development_neighbor_diagnostics_v1.json"
payload = json.dumps(report, indent=2) + "\n"

if report_path.exists():
    assert report_path.read_text(encoding="utf-8") == payload
    report_status = "EXISTING_VERIFIED"
else:
    temporary = report_path.with_suffix(".json.tmp")
    temporary.write_text(payload, encoding="utf-8")
    os.replace(temporary, report_path)
    report_status = "NEW_SAVED"

print("STAGE 11 — DEVELOPMENT NEIGHBOR DIAGNOSTICS")
print("Development records:", len(evaluation))
print(f"Top-1 neighbor accuracy: {accuracy:.4%}")
print(f"Macro F1: {macro_f1:.4f}")

print("\nPER-CLASS ACCURACY")
display(pd.DataFrame(class_results))

print("\nCONFUSION MATRIX — ROWS: TRUE, COLUMNS: PREDICTED")
display(pd.DataFrame(
    confusion,
    index=class_names,
    columns=class_names
))

print("\nSUPPORT-THRESHOLD DIAGNOSTICS")
display(pd.DataFrame(threshold_results))

print("\nReport:", report_status)
print("Saved path:", report_path)
print("Correction performed: NO")
print("Threshold selected: NO")
print("Training reference labels accessed: NO")
print("Holdout reference labels accessed: NO")
print("\nSTATUS: DEVELOPMENT DIAGNOSTICS PASSED")


STAGE 11 — DEVELOPMENT NEIGHBOR DIAGNOSTICS
Development records: 1800
Top-1 neighbor accuracy: 88.3333%
Macro F1: 0.8831

PER-CLASS ACCURACY


,class_id,class_name,records,correct,accuracy
0,0,World,450,390,0.866667
1,1,Sports,450,439,0.975556
2,2,Business,450,384,0.853333
3,3,Sci/Tech,450,377,0.837778



CONFUSION MATRIX — ROWS: TRUE, COLUMNS: PREDICTED


,World,Sports,Business,Sci/Tech
World,390,15,32,13
Sports,7,439,3,1
Business,11,3,384,52
Sci/Tech,27,6,40,377



SUPPORT-THRESHOLD DIAGNOSTICS


,minimum_top_support,selected_records,coverage,accuracy_if_selected
0,0.5,1690,0.938889,0.910059
1,0.6,1518,0.843333,0.931489
2,0.7,1337,0.742778,0.949140
3,0.8,1112,0.617778,0.965827
4,0.9,768,0.426667,0.984375



Report: NEW_SAVED
Saved path: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_evidence_v1/development_neighbor_diagnostics_v1.json
Correction performed: NO
Threshold selected: NO
Training reference labels accessed: NO
Holdout reference labels accessed: NO

STATUS: DEVELOPMENT DIAGNOSTICS PASSED


In [8]:

import os
import json
import hashlib
import numpy as np
import pandas as pd

EVIDENCE_DIR = BASE / "outputs" / "s_slte_evidence_v1"

with (EVIDENCE_DIR / "evidence_manifest_v1.json").open(
    encoding="utf-8"
) as f:
    evidence_manifest = json.load(f)

assert evidence_manifest["k"] == 20
assert evidence_manifest["neighbor_pool"] == "CLEAN_ANCHOR_ONLY"

def load_verified_evidence(partition, expected_frame):
    path = EVIDENCE_DIR / f"{partition}_neighbor_evidence_v1.csv"

    assert sha256_file(path) == (
        evidence_manifest["partitions"][partition]["sha256"]
    )

    data = pd.read_csv(path).sort_values(
        "record_id"
    ).reset_index(drop=True)

    expected_ids = (
        expected_frame["record_id"].sort_values()
        .to_numpy(dtype=np.int64)
    )

    assert np.array_equal(
        data["record_id"].to_numpy(dtype=np.int64),
        expected_ids
    )
    assert data["has_positive_similarity"].all()
    assert data["neighbor_top_label"].isin(range(4)).all()
    assert data["neighbor_top_support"].between(0, 1).all()

    return data, path

dev_evidence, dev_path = load_verified_evidence(
    "development", dev_df
)

training_evidence, training_path = load_verified_evidence(
    "training", training_df
)

dev_evaluation = dev_evidence.merge(
    dev_df[["record_id", "label"]],
    on="record_id",
    validate="one_to_one"
)

training_evaluation = training_evidence.merge(
    training_df[["record_id", "llm_label"]],
    on="record_id",
    validate="one_to_one",
    suffixes=("", "_source")
)

assert np.array_equal(
    training_evaluation["llm_label"],
    training_evaluation["llm_label_source"]
)

dev_correct = (
    dev_evaluation["neighbor_top_label"].to_numpy()
    == dev_evaluation["label"].to_numpy()
)

train_conflict = (
    training_evaluation["neighbor_top_label"].to_numpy()
    != training_evaluation["llm_label"].to_numpy()
)

assert len(dev_correct) == 1800
assert len(train_conflict) == 13621
assert int(dev_correct.sum()) == 1590
assert int(train_conflict.sum()) == 2590

dev_support = dev_evaluation[
    "neighbor_top_support"
].to_numpy()

train_support = training_evaluation[
    "neighbor_top_support"
].to_numpy()

threshold_rows = []

for threshold in [0.50, 0.60, 0.70, 0.80, 0.90]:
    dev_selected = dev_support >= threshold
    train_selected = train_support >= threshold

    dev_count = int(dev_selected.sum())
    dev_errors = int((dev_selected & ~dev_correct).sum())

    train_conflicts = int(
        (train_selected & train_conflict).sum()
    )

    threshold_rows.append({
        "threshold": threshold,
        "development_selected": dev_count,
        "development_incorrect": dev_errors,
        "development_selected_error_rate": (
            round(dev_errors / dev_count, 6)
            if dev_count else None
        ),
        "training_high_support_conflicts": train_conflicts,
        "training_conflict_coverage": round(
            train_conflicts / 2590, 6
        )
    })

threshold_table = pd.DataFrame(threshold_rows)

class_rows = []

for class_id, class_name in enumerate(
    ["World", "Sports", "Business", "Sci/Tech"]
):
    mask = (
        training_evaluation["llm_label"].to_numpy()
        == class_id
    )

    class_rows.append({
        "llm_class": class_name,
        "total": int(mask.sum()),
        "neighbor_conflicts": int(
            (mask & train_conflict).sum()
        ),
        "conflict_rate": round(
            float(train_conflict[mask].mean()), 6
        )
    })

class_table = pd.DataFrame(class_rows)

report = {
    "protocol_version": "v1",
    "analysis": "HIGH_CONFIDENCE_NEIGHBOR_CONFLICT_RISK",
    "k": 20,
    "development_records": 1800,
    "training_records": 13621,
    "development_correct_neighbor_predictions": 1590,
    "training_neighbor_llm_conflicts": 2590,
    "thresholds_exploratory_only": True,
    "threshold_selected": False,
    "correction_performed": False,
    "development_labels_used_for_diagnostics": True,
    "training_reference_labels_accessed": False,
    "holdout_reference_labels_accessed": False,
    "threshold_diagnostics": threshold_rows,
    "conflicts_by_llm_class": class_rows,
    "source_sha256": {
        "development": sha256_file(dev_path),
        "training": sha256_file(training_path)
    }
}

report_path = (
    EVIDENCE_DIR / "high_confidence_conflict_audit_v1.json"
)

payload = json.dumps(report, indent=2) + "\n"

if report_path.exists():
    assert report_path.read_text(
        encoding="utf-8"
    ) == payload
    status = "EXISTING_VERIFIED"
else:
    temporary = report_path.with_suffix(".json.tmp")
    temporary.write_text(payload, encoding="utf-8")
    os.replace(temporary, report_path)
    status = "NEW_SAVED"

print("STAGE 11 — HIGH-CONFIDENCE CONFLICT AUDIT")

print("\nSUPPORT THRESHOLD ANALYSIS")
display(threshold_table)

print("\nTRAINING CONFLICTS BY LLM CLASS")
display(class_table)

print("\nTraining conflicts:", int(train_conflict.sum()))
print("Development errors:", int((~dev_correct).sum()))
print("Report:", status)
print("Threshold selected: NO")
print("Correction performed: NO")
print("Training reference labels accessed: NO")
print("Holdout reference labels accessed: NO")
print("\nSTATUS: CONFLICT RISK AUDIT PASSED")


STAGE 11 — HIGH-CONFIDENCE CONFLICT AUDIT

SUPPORT THRESHOLD ANALYSIS


,threshold,development_selected,development_incorrect,development_selected_error_rate,training_high_support_conflicts,training_conflict_coverage
0,0.5,1690,152,0.089941,2057,0.794208
1,0.6,1518,104,0.068511,1519,0.586486
2,0.7,1337,68,0.050860,1097,0.423552
3,0.8,1112,38,0.034173,705,0.272201
4,0.9,768,12,0.015625,293,0.113127



TRAINING CONFLICTS BY LLM CLASS


,llm_class,total,neighbor_conflicts,conflict_rate
0,World,4199,1031,0.245535
1,Sports,3532,170,0.048131
2,Business,4015,1209,0.301121
3,Sci/Tech,1875,180,0.096000



Training conflicts: 2590
Development errors: 210
Report: NEW_SAVED
Threshold selected: NO
Correction performed: NO
Training reference labels accessed: NO
Holdout reference labels accessed: NO

STATUS: CONFLICT RISK AUDIT PASSED


In [9]:

import os
import json
import numpy as np
import pandas as pd

EVIDENCE_DIR = BASE / "outputs" / "s_slte_evidence_v1"
THRESHOLDS = [0.70, 0.80, 0.90]
CLASS_NAMES = ["World", "Sports", "Business", "Sci/Tech"]

with (EVIDENCE_DIR / "evidence_manifest_v1.json").open(
    encoding="utf-8"
) as f:
    evidence_manifest = json.load(f)

def load_evidence(partition):
    path = EVIDENCE_DIR / f"{partition}_neighbor_evidence_v1.csv"

    assert sha256_file(path) == (
        evidence_manifest["partitions"][partition]["sha256"]
    )

    data = pd.read_csv(path)
    assert data["record_id"].is_unique
    assert data["has_positive_similarity"].all()

    return data

dev_evidence = load_evidence("development")
train_evidence = load_evidence("training")

dev = dev_evidence.merge(
    dev_df[["record_id", "label"]],
    on="record_id",
    validate="one_to_one"
)

train = train_evidence.copy()

assert len(dev) == 1800
assert len(train) == 13621
assert "label" not in train.columns

source_labels = training_df.set_index("record_id")["llm_label"]
assert np.array_equal(
    train["llm_label"].to_numpy(),
    source_labels.loc[train["record_id"]].to_numpy()
)

assert dev["neighbor_top_label"].isin(range(4)).all()
assert train["neighbor_top_label"].isin(range(4)).all()

def wilson_upper_bound(errors, n, z=1.959963984540054):
    if n == 0:
        return None

    p = errors / n
    denominator = 1.0 + z**2 / n

    center = (
        p + z**2 / (2 * n)
    ) / denominator

    radius = (
        z * np.sqrt(
            p * (1 - p) / n + z**2 / (4 * n**2)
        )
    ) / denominator

    return round(float(min(1.0, center + radius)), 6)

dev_prediction = dev["neighbor_top_label"].to_numpy()
dev_truth = dev["label"].to_numpy()
dev_support = dev["neighbor_top_support"].to_numpy()

train_prediction = train["neighbor_top_label"].to_numpy()
train_llm = train["llm_label"].to_numpy()
train_support = train["neighbor_top_support"].to_numpy()

train_conflict = train_prediction != train_llm
assert int(train_conflict.sum()) == 2590

rows = []

for threshold in THRESHOLDS:
    for class_id, class_name in enumerate(CLASS_NAMES):
        selected_dev = (
            (dev_prediction == class_id)
            & (dev_support >= threshold)
        )

        selected_train = (
            (train_prediction == class_id)
            & (train_support >= threshold)
            & train_conflict
        )

        n = int(selected_dev.sum())
        errors = int(
            (selected_dev & (dev_prediction != dev_truth)).sum()
        )

        candidates = int(selected_train.sum())

        rows.append({
            "threshold": threshold,
            "proposed_class": class_name,
            "development_selected": n,
            "development_errors": errors,
            "development_error_rate": (
                round(errors / n, 6) if n else None
            ),
            "wilson_95_upper_error_bound": (
                wilson_upper_bound(errors, n)
            ),
            "training_conflict_candidates": candidates
        })

risk_table = pd.DataFrame(rows)

# Validate reported class totals against overall audit.
for threshold in THRESHOLDS:
    subset = risk_table.loc[
        risk_table["threshold"] == threshold
    ]

    selected_dev = int((dev_support >= threshold).sum())
    selected_train = int(
        ((train_support >= threshold) & train_conflict).sum()
    )

    assert subset["development_selected"].sum() == selected_dev
    assert subset["training_conflict_candidates"].sum() == (
        selected_train
    )

    print(f"\nSUPPORT THRESHOLD >= {threshold:.2f}")
    display(
        subset.drop(columns=["threshold"]).reset_index(drop=True)
    )

report = {
    "protocol_version": "v1",
    "analysis": "CLASS_CONDITIONAL_SELECTIVE_RISK",
    "k": 20,
    "thresholds": THRESHOLDS,
    "confidence_interval": "ONE_SIDED_95_PERCENT_WILSON_UPPER",
    "class_conditional_results": rows,
    "threshold_selected": False,
    "label_correction_performed": False,
    "development_reference_labels_used": True,
    "training_reference_labels_accessed": False,
    "holdout_reference_labels_accessed": False,
    "limitation": (
        "Development neighborhood prediction error is not "
        "the same as expected training label-correction error."
    ),
    "source_sha256": {
        "development": sha256_file(
            EVIDENCE_DIR / "development_neighbor_evidence_v1.csv"
        ),
        "training": sha256_file(
            EVIDENCE_DIR / "training_neighbor_evidence_v1.csv"
        )
    }
}

report_path = (
    EVIDENCE_DIR / "class_conditional_risk_audit_v1.json"
)

payload = json.dumps(report, indent=2) + "\n"

if report_path.exists():
    assert report_path.read_text(
        encoding="utf-8"
    ) == payload
    status = "EXISTING_VERIFIED"
else:
    temporary = report_path.with_suffix(".json.tmp")
    temporary.write_text(payload, encoding="utf-8")
    os.replace(temporary, report_path)
    status = "NEW_SAVED"

print("\nSTAGE 11 — CLASS-CONDITIONAL RISK AUDIT")
print("Development records:", len(dev))
print("Training records:", len(train))
print("Total neighbor–LLM conflicts:", int(train_conflict.sum()))
print("Report:", status)
print("Saved path:", report_path)
print("Threshold selected: NO")
print("Corrections performed: NO")
print("Training reference labels accessed: NO")
print("Holdout reference labels accessed: NO")
print("\nSTATUS: CLASS-CONDITIONAL RISK AUDIT PASSED")



SUPPORT THRESHOLD >= 0.70


,proposed_class,development_selected,development_errors,development_error_rate,wilson_95_upper_error_bound,training_conflict_candidates
0,World,329,7,0.021277,0.043260,30
1,Sports,425,12,0.028235,0.048698,66
2,Business,295,28,0.094915,0.133759,184
3,Sci/Tech,288,21,0.072917,0.108893,817



SUPPORT THRESHOLD >= 0.80


,proposed_class,development_selected,development_errors,development_error_rate,wilson_95_upper_error_bound,training_conflict_candidates
0,World,280,4,0.014286,0.036149,10
1,Sports,407,8,0.019656,0.038302,49
2,Business,215,16,0.074419,0.117455,96
3,Sci/Tech,210,10,0.047619,0.085423,550



SUPPORT THRESHOLD >= 0.90


,proposed_class,development_selected,development_errors,development_error_rate,wilson_95_upper_error_bound,training_conflict_candidates
0,World,218,1,0.004587,0.025521,2
1,Sports,333,6,0.018018,0.038746,32
2,Business,107,4,0.037383,0.092200,38
3,Sci/Tech,110,1,0.009091,0.049706,221



STAGE 11 — CLASS-CONDITIONAL RISK AUDIT
Development records: 1800
Training records: 13621
Total neighbor–LLM conflicts: 2590
Report: NEW_SAVED
Saved path: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_evidence_v1/class_conditional_risk_audit_v1.json
Threshold selected: NO
Corrections performed: NO
Training reference labels accessed: NO
Holdout reference labels accessed: NO

STATUS: CLASS-CONDITIONAL RISK AUDIT PASSED


In [2]:

from pathlib import Path
import json
import hashlib

import pandas as pd
import numpy as np
from google.colab import drive

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")

if not (BASE / "configs").exists():
    drive.mount("/content/drive")

assert BASE.exists(), "Research workspace not found."

EVIDENCE_DIR = BASE / "outputs/s_slte_evidence_v1"

with (
    BASE / "outputs/llm_main_v1/full_annotation_audit_v1.json"
).open(encoding="utf-8") as f:
    annotation_audit = json.load(f)

with (
    EVIDENCE_DIR / "evidence_manifest_v1.json"
).open(encoding="utf-8") as f:
    evidence_manifest = json.load(f)

with (
    BASE / "configs/s_slte_embedding_protocol_v1.json"
).open(encoding="utf-8") as f:
    embedding_protocol = json.load(f)

MODEL_ID = embedding_protocol["model_id"]
MODEL_REVISION = embedding_protocol["model_revision_sha"]

def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

assert annotation_audit["status"] == "FULL_CHECKPOINT_AUDIT_PASSED"
assert annotation_audit["total_records"] == 15621
assert evidence_manifest["model_id"] == MODEL_ID
assert evidence_manifest["model_revision_sha"] == MODEL_REVISION

for partition in ["development", "training"]:
    path = EVIDENCE_DIR / f"{partition}_neighbor_evidence_v1.csv"
    assert path.exists()
    assert sha256_file(path) == (
        evidence_manifest["partitions"][partition]["sha256"]
    )

print("STAGE 11 — RUNTIME RECOVERY PASSED")
print("Workspace: VERIFIED")
print("Annotation audit: VERIFIED")
print("Embedding protocol: VERIFIED")
print("Neighborhood evidence: VERIFIED")
print("Model:", MODEL_ID)
print("GPU inference required: NO")
print("Previous research files modified: NO")
print("STATUS: READY TO RERUN STEP 10")


Mounted at /content/drive
STAGE 11 — RUNTIME RECOVERY PASSED
Workspace: VERIFIED
Annotation audit: VERIFIED
Embedding protocol: VERIFIED
Neighborhood evidence: VERIFIED
Model: sentence-transformers/all-MiniLM-L6-v2
GPU inference required: NO
Previous research files modified: NO
STATUS: READY TO RERUN STEP 10


In [3]:

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

SCORE_DIR = BASE / "outputs" / "s_slte_scores_v1"
SCORE_DIR.mkdir(parents=True, exist_ok=True)

source_path = (
    BASE / "outputs/llm_main_v1/training_annotations_v1.csv"
)
evidence_path = (
    EVIDENCE_DIR / "training_neighbor_evidence_v1.csv"
)

assert sha256_file(source_path) == (
    annotation_audit["exports"]["training"]["sha256"]
)
assert sha256_file(evidence_path) == (
    evidence_manifest["partitions"]["training"]["sha256"]
)

evidence = (
    pd.read_csv(evidence_path)
    .sort_values("record_id")
    .reset_index(drop=True)
)

annotations = (
    pd.read_csv(source_path)
    .sort_values("record_id")
    .reset_index(drop=True)
)

assert len(evidence) == len(annotations) == 13621
assert np.array_equal(
    evidence["record_id"].to_numpy(),
    annotations["record_id"].to_numpy()
)
assert np.array_equal(
    evidence["llm_label"].to_numpy(),
    annotations["llm_label"].to_numpy()
)
assert evidence["has_positive_similarity"].all()

class_columns = [f"support_class_{i}" for i in range(4)]
support = evidence[class_columns].to_numpy(dtype=np.float64)
llm_labels = annotations["llm_label"].to_numpy(dtype=int)

assert np.isfinite(support).all()
assert np.all(support >= 0)
assert np.allclose(support.sum(axis=1), 1.0, atol=1e-6)
assert np.isin(llm_labels, range(4)).all()

row_indices = np.arange(len(evidence))

# Semantic support assigned to the original LLM label.
label_trust = support[row_indices, llm_labels]

# Strongest semantic label supported by clean anchors.
neighbor_labels = support.argmax(axis=1)
neighbor_support = support.max(axis=1)

conflict = neighbor_labels != llm_labels

# Zero for agreement; positive when an alternative label is stronger.
support_gap = np.maximum(
    neighbor_support - label_trust, 0.0
)

assert np.allclose(
    label_trust,
    evidence["llm_label_support"].to_numpy(),
    atol=1e-6
)
assert np.array_equal(
    neighbor_labels,
    evidence["neighbor_top_label"].to_numpy()
)
assert int(conflict.sum()) == 2590
assert np.allclose(support_gap[~conflict], 0.0, atol=1e-6)

scores = pd.DataFrame({
    "record_id": evidence["record_id"].to_numpy(dtype=np.int64),
    "llm_label": llm_labels,
    "semantic_label_trustworthiness": label_trust,
    "neighbor_proposed_label": neighbor_labels,
    "neighbor_top_support": neighbor_support,
    "semantic_support_gap": support_gap,
    "llm_neighbor_conflict": conflict,
    "neighbor_top_margin": evidence[
        "neighbor_support_margin"
    ].to_numpy(),
    "max_cosine_similarity": evidence[
        "max_cosine_similarity"
    ].to_numpy()
})

assert scores["record_id"].is_unique
assert "label" not in scores.columns
assert len(scores) == 13621

# Exploratory candidate counts; no thresholds are selected.
diagnostic_rows = []

for min_support in [0.70, 0.80, 0.90]:
    for min_gap in [0.0, 0.2, 0.4]:
        selected = (
            conflict
            & (neighbor_support >= min_support)
            & (support_gap >= min_gap)
        )

        row = {
            "minimum_neighbor_support": min_support,
            "minimum_support_gap": min_gap,
            "candidate_count": int(selected.sum())
        }

        for class_id, class_name in enumerate(
            ["World", "Sports", "Business", "SciTech"]
        ):
            row[f"proposed_{class_name}"] = int(
                (selected & (neighbor_labels == class_id)).sum()
            )

        diagnostic_rows.append(row)

diagnostics = pd.DataFrame(diagnostic_rows)

expected_base_counts = {
    0.70: 1097,
    0.80: 705,
    0.90: 293
}

for threshold, expected_count in expected_base_counts.items():
    row = diagnostics.loc[
        (diagnostics["minimum_neighbor_support"] == threshold)
        & (diagnostics["minimum_support_gap"] == 0.0)
    ]

    assert len(row) == 1
    assert int(row.iloc[0]["candidate_count"]) == expected_count

output_path = (
    SCORE_DIR / "training_semantic_trust_scores_v1.csv"
)

csv_content = scores.to_csv(
    index=False,
    float_format="%.8f",
    lineterminator="\n"
)

if output_path.exists():
    assert output_path.read_text(
        encoding="utf-8"
    ) == csv_content
    score_status = "EXISTING_VERIFIED"
else:
    temporary = output_path.with_suffix(".csv.tmp")
    temporary.write_text(csv_content, encoding="utf-8")
    os.replace(temporary, output_path)
    score_status = "NEW_SAVED"

score_manifest = {
    "protocol_version": "v1",
    "component": "UNCALIBRATED_SEMANTIC_LABEL_SUPPORT",
    "model_id": MODEL_ID,
    "model_revision_sha": MODEL_REVISION,
    "neighbor_pool": "CLEAN_ANCHOR_ONLY",
    "k": 20,
    "records": 13621,
    "conflicts": 2590,
    "score_definition": (
        "Weighted semantic support of the original LLM label"
    ),
    "gap_definition": (
        "Maximum neighbor class support minus LLM label support"
    ),
    "thresholds_selected": False,
    "correction_performed": False,
    "training_reference_labels_accessed": False,
    "holdout_reference_labels_accessed": False,
    "source_sha256": {
        "llm_annotations": sha256_file(source_path),
        "neighbor_evidence": sha256_file(evidence_path)
    },
    "score_file_sha256": sha256_file(output_path),
    "candidate_diagnostics": diagnostic_rows,
    "statistical_note": (
        "Earlier z=1.95996398454 Wilson upper endpoints "
        "are upper limits of two-sided 95% intervals, "
        "equivalently one-sided 97.5% bounds."
    )
}

manifest_path = SCORE_DIR / "semantic_score_manifest_v1.json"
manifest_content = json.dumps(
    score_manifest, indent=2
) + "\n"

if manifest_path.exists():
    assert manifest_path.read_text(
        encoding="utf-8"
    ) == manifest_content
    manifest_status = "EXISTING_VERIFIED"
else:
    temporary = manifest_path.with_suffix(".json.tmp")
    temporary.write_text(
        manifest_content, encoding="utf-8"
    )
    os.replace(temporary, manifest_path)
    manifest_status = "NEW_SAVED"

print("STAGE 11 — SEMANTIC TRUSTWORTHINESS SCORES")
print("Training records:", len(scores))
print("Neighbor–LLM agreements:", int((~conflict).sum()))
print("Neighbor–LLM conflicts:", int(conflict.sum()))

print("\nEXPLORATORY CONFLICT CANDIDATES")
display(diagnostics)

print("\nSEMANTIC TRUSTWORTHINESS SUMMARY")
display(
    scores.groupby("llm_neighbor_conflict")[
        "semantic_label_trustworthiness"
    ].agg(["count", "mean", "median", "min", "max"])
)

print("\nScores:", score_status)
print("Manifest:", manifest_status)
print("Saved directory:", SCORE_DIR)
print("Threshold selected: NO")
print("Correction performed: NO")
print("Training reference labels accessed: NO")
print("Holdout reference labels accessed: NO")
print("\nSTATUS: SEMANTIC TRUSTWORTHINESS SCORES VERIFIED")


STAGE 11 — SEMANTIC TRUSTWORTHINESS SCORES
Training records: 13621
Neighbor–LLM agreements: 11031
Neighbor–LLM conflicts: 2590

EXPLORATORY CONFLICT CANDIDATES


,minimum_neighbor_support,minimum_support_gap,candidate_count,proposed_World,proposed_Sports,proposed_Business,proposed_SciTech
0,0.7,0.0,1097,30,66,184,817
1,0.7,0.2,1097,30,66,184,817
2,0.7,0.4,1097,30,66,184,817
3,0.8,0.0,705,10,49,96,550
4,0.8,0.2,705,10,49,96,550
5,0.8,0.4,705,10,49,96,550
6,0.9,0.0,293,2,32,38,221
7,0.9,0.2,293,2,32,38,221
8,0.9,0.4,293,2,32,38,221



SEMANTIC TRUSTWORTHINESS SUMMARY


,count,mean,median,min,max
llm_neighbor_conflict,,,,,
False,11031,0.848687,0.901822,0.282547,1.000000
True,2590,0.207993,0.196306,0.000000,0.499168



Scores: NEW_SAVED
Manifest: NEW_SAVED
Saved directory: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_scores_v1
Threshold selected: NO
Correction performed: NO
Training reference labels accessed: NO
Holdout reference labels accessed: NO

STATUS: SEMANTIC TRUSTWORTHINESS SCORES VERIFIED


In [4]:

import json
import hashlib
from pathlib import Path

import pandas as pd

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")

with (BASE / "configs/holdout_partition_manifest_v1.json").open(
    encoding="utf-8"
) as f:
    frozen_manifest = json.load(f)

with (BASE / "configs/annotation_protocol_v1.json").open(
    encoding="utf-8"
) as f:
    llm_protocol = json.load(f)

with (BASE / "configs/llm_pilot_manifest_v2.json").open(
    encoding="utf-8"
) as f:
    pilot_manifest = json.load(f)

# Load text only. No development reference labels enter inference.
dev_annotation_inputs = pd.read_csv(
    BASE / "data/splits/development.csv",
    usecols=["record_id", "text"]
).sort_values("record_id").reset_index(drop=True)

spec = frozen_manifest["partitions"]["development"]

def fingerprint_ids(ids):
    return hashlib.sha256(
        "\n".join(
            str(int(x)) for x in sorted(ids)
        ).encode("utf-8")
    ).hexdigest()

def fingerprint_text(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values("record_id").itertuples(index=False):
        digest.update(str(int(row.record_id)).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(row.text).encode("utf-8"))
        digest.update(b"\0")

    return digest.hexdigest()

assert len(dev_annotation_inputs) == spec["count"] == 1800
assert dev_annotation_inputs["record_id"].is_unique
assert dev_annotation_inputs["text"].notna().all()
assert dev_annotation_inputs.columns.tolist() == [
    "record_id", "text"
]
assert fingerprint_ids(
    dev_annotation_inputs["record_id"]
) == spec["id_sha256"]
assert fingerprint_text(
    dev_annotation_inputs
) == spec["raw_text_sha256"]

DEV_LLM_MODEL_ID = pilot_manifest["model_id"]
DEV_LLM_REVISION = pilot_manifest["model_revision_sha"]
DEV_SYSTEM_PROMPT = llm_protocol["system_prompt"]

DEV_PROMPT_SHA256 = hashlib.sha256(
    DEV_SYSTEM_PROMPT.encode("utf-8")
).hexdigest()

assert DEV_LLM_MODEL_ID == "meta-llama/Llama-3.1-8B-Instruct"
assert DEV_LLM_REVISION == (
    "0e9e39f249a16976918f6564b8830bc894c89659"
)
assert DEV_PROMPT_SHA256 == (
    "77a8d266f6f9d3e2dcdbeb906182e0d64f0836190fe84617a638d3ab3c5ed794"
)
assert DEV_PROMPT_SHA256 == pilot_manifest["prompt_sha256"]

DEV_ANNOTATION_DIR = (
    BASE / "outputs/llm_development_v1"
)
DEV_ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

dev_manifest = {
    "experiment_id": "s_slte_development_annotation_v1",
    "purpose": "DEVELOPMENT_CORRECTION_CALIBRATION",
    "partition": "development",
    "partition_manifest_version": "v1",
    "expected_records": 1800,
    "development_id_sha256": spec["id_sha256"],
    "development_text_sha256": spec["raw_text_sha256"],
    "model_id": DEV_LLM_MODEL_ID,
    "model_revision_sha": DEV_LLM_REVISION,
    "prompt_sha256": DEV_PROMPT_SHA256,
    "quantization": "4-bit NF4",
    "text_policy": "ORIGINAL_TEXT_UNMODIFIED",
    "generation": {
        "do_sample": False,
        "max_new_tokens": 32
    },
    "reference_labels_in_inference": False
}

manifest_path = (
    DEV_ANNOTATION_DIR / "development_annotation_manifest_v1.json"
)

if manifest_path.exists():
    with manifest_path.open(encoding="utf-8") as f:
        assert json.load(f) == dev_manifest

    status = "EXISTING_VERIFIED"
else:
    with manifest_path.open("x", encoding="utf-8") as f:
        json.dump(dev_manifest, f, indent=2)

    status = "NEW_SAVED"

print("STAGE 11 — DEVELOPMENT ANNOTATION PREPARATION")
print("Records:", len(dev_annotation_inputs))
print("Frozen IDs: VERIFIED")
print("Original text fingerprint: VERIFIED")
print("Model:", DEV_LLM_MODEL_ID)
print("Model revision:", DEV_LLM_REVISION)
print("Prompt fingerprint: VERIFIED")
print("Manifest:", status)
print("Development reference labels loaded: NO")
print("Training reference labels loaded: NO")
print("Holdout reference labels loaded: NO")
print("New inference performed: NO")
print("\nSTATUS: DEVELOPMENT ANNOTATION READY")


STAGE 11 — DEVELOPMENT ANNOTATION PREPARATION
Records: 1800
Frozen IDs: VERIFIED
Original text fingerprint: VERIFIED
Model: meta-llama/Llama-3.1-8B-Instruct
Model revision: 0e9e39f249a16976918f6564b8830bc894c89659
Prompt fingerprint: VERIFIED
Manifest: NEW_SAVED
Development reference labels loaded: NO
Training reference labels loaded: NO
Holdout reference labels loaded: NO
New inference performed: NO

STATUS: DEVELOPMENT ANNOTATION READY
